# Medium creation and completion

## 1. Imports, Directores, File loading and community creation

### 1.1 Imports

In [1]:
import os
import sys
sys.path.insert(0, '/home/emma/Dokumente/thesis')

import pandas as pd
from micom import Community
from micom.media import complete_medium
from functions import *

### 1.2 Directories and files

In [2]:
media_raw_dir = "/home/emma/Dokumente/thesis/media_creation/raw_csv/"
media_dir = "/home/emma/Dokumente/thesis/media_creation/created_media"
model_dir = "/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/final"

In [3]:
translate2bigg = pd.read_csv("/home/emma/Dokumente/thesis/media_creation/translate_tobigg.csv", delimiter=",")
taxonomy_df = pd.read_csv("/home/emma/Dokumente/thesis/communities/SynComs_taxonomy.csv")

In [4]:
met_class = "/home/emma/Dokumente/thesis/completed_metabolite_classification.csv"
met_class_df = pd.read_csv(met_class)
met_class_dict = dict(zip(met_class_df["Reaction"], met_class_df["Category"]))

In [5]:
# Load and store all media 
AF7 = pd.read_csv(os.path.join(media_dir, "media_barley - 7dpi_AF_mock.csv"))

### 1.3 Functions
All other functions can be found in the `functions.py` file

In [6]:
def create_scaled_media(raw_data_csv):
    df_averaged = raw_data_csv.groupby("Compound", as_index=False)["Resp.Ratio"].mean()
    df_sorted = df_averaged.sort_values(by="Resp.Ratio", ascending=False) 
    compounds_rela = dict(zip(df_sorted["Compound"], df_sorted["Resp.Ratio"]))
    max_ab = max(compounds_rela.values())
    if max_ab == 0:
        return {comp_id: 0.1 for comp_id in compounds_rela}
    
    scale_fac = 1000 / max_ab
    
    compounds_scaled = {}
    for comp_id, abundance in compounds_rela.items():
        scaled_val = abundance * scale_fac
        compounds_scaled[comp_id] = scaled_val
        
    return compounds_scaled

In [7]:
# Translate compound names to BiGG exchange reaction IDs and aggregate abundances per reaction.
def translate_to_bigg(scaled_dict, translate2bigg_df):
    mapping_dict = {
        str(name).strip(): str(reaction).strip()
        for name, reaction in zip(
            translate2bigg_df["Original Compound Name"],
            translate2bigg_df["BiGG Exchange Reaction"],
        )
        if pd.notna(name) and pd.notna(reaction)
    }

    translated_dict = {}
    for comp_name, abundance in scaled_dict.items():
        bigg_id = mapping_dict.get(str(comp_name).strip())
        if not bigg_id or bigg_id == "Exclude" or bigg_id == "nan":
            continue

        translated_dict[bigg_id] = translated_dict.get(bigg_id, 0.0) + abundance

    return translated_dict

In [8]:
def create_class_media(translated_dict):
    tier_media = {}
    for comp_id, scal_abund in translated_dict.items():
        if scal_abund <= 0.1:
            tier_media[comp_id] = 0.1
        elif 0.1 < scal_abund <= 10:
            tier_media[comp_id] = 10
        elif 10 < scal_abund <= 100:
            tier_media[comp_id] = 100
        elif 100 < scal_abund <= 500:
            tier_media[comp_id] = 500
        else: 
            tier_media[comp_id] = 1000 
            
    return tier_media

In [9]:
def create_medium_total(raw_data_csv, translate2bigg, media_id, media_dir):
    scaled = create_scaled_media(raw_data_csv)
    translated = translate_to_bigg(scaled, translate2bigg)
    classed = create_class_media(translated)
    save_media(classed, media_id, media_dir)

### 1.4 Create Communities 

In [10]:
HvSC1_df = taxonomy_df[(taxonomy_df["Syncom"] == "HvSC1") | (taxonomy_df["Syncom"] == "Both")]
HvSC1_df = HvSC1_df[["Strain ID","Family"]].rename(columns={'Strain ID': 'id', 'Family': 'family'}).copy()
HvSC1_df["id"] = HvSC1_df["id"].astype(str)
HvSC1_df["abundance"] = 1
HvSC1_df["file"] = HvSC1_df['id'].apply(lambda x: f"{model_dir}/{x}_or_mb1_mdr_rdr_dp_mb2_lib_bz_fix.xml")

In [11]:
HvSC2_df = taxonomy_df[(taxonomy_df["Syncom"] == "HvSC2") | (taxonomy_df["Syncom"] == "Both")]
HvSC2_df = HvSC2_df[["Strain ID","Family"]].rename(columns={'Strain ID': 'id', 'Family': 'family'}).copy()
HvSC2_df["id"] = HvSC2_df["id"].astype(str)
HvSC2_df["abundance"] = 1
HvSC2_df["file"] = HvSC2_df['id'].apply(lambda x: f"{model_dir}/{x}_or_mb1_mdr_rdr_dp_mb2_lib_bz_fix.xml")

In [12]:
HvSC1 = Community(HvSC1_df, id = "HvSC1", name = "HvSC1")

Output()

In [13]:
HvSC2 = Community(HvSC2_df, id = "HvSC2", name = "HvSC2")

Output()

## 2. Create Medium

### 2.1 Clean raw metabolomics file and create tiered medium

In [ ]:
for file in os.listdir(media_raw_dir):
    if file.endswith(".csv"):
        media_path = os.path.join(media_raw_dir, file)
        raw_data = pd.read_csv(media_path, delimiter=",", thousands=",")
        media_id = os.path.splitext(file)[0]
        create_medium_total(raw_data, translate2bigg, media_id, media_dir)

### 2.2 Complete medium with MICOM's 'complete_medium()'

In [15]:
AF7 = pd.read_csv(os.path.join(media_dir, "media_barley - 7dpi_AF_mock.csv"), header = None)
AF7_s = compute_medium_series(AF7)

In [18]:
min_com_growth = 1
min_indi_growth_sc1 = 1 / len(HvSC1.taxa)
min_indi_growth_sc2 = 1 / len(HvSC2.taxa)

In [19]:
af_c1 = complete_medium(HvSC1, AF7_s, min_com_growth, min_indi_growth_sc1, minimize_components=True)

[07/28/26 11:06:43] WARNING  the MIP version of minimal media is extremely slow for models that large   ]8;id=639864;file:///home/emma/miniconda3/envs/micom/lib/python3.10/site-packages/micom/media.py\media.py]8;;\:]8;id=792181;file:///home/emma/miniconda3/envs/micom/lib/python3.10/site-packages/micom/media.py#72\72]8;;\
                             :(                                                                                    

In [20]:
af_c2 = complete_medium(HvSC2, AF7_s, min_com_growth, min_indi_growth_sc2, minimize_components=True)

[07/28/26 11:09:30] WARNING  the MIP version of minimal media is extremely slow for models that large   ]8;id=745272;file:///home/emma/miniconda3/envs/micom/lib/python3.10/site-packages/micom/media.py\media.py]8;;\:]8;id=780870;file:///home/emma/miniconda3/envs/micom/lib/python3.10/site-packages/micom/media.py#72\72]8;;\
                             :(                                                                                    

In [21]:
# Merge af_c1 and af_c2: if a reaction appears in both, keep the higher flux bound.
combined_afmedium = af_c1.copy()
for ex_r, flux in af_c2.items():
    if ex_r not in combined_afmedium.keys():
        combined_afmedium[ex_r] = flux
    if ex_r in combined_afmedium.keys():
        if combined_afmedium[ex_r] < flux:
            combined_afmedium[ex_r] = flux
combined_afmedium_d = combined_afmedium.to_dict()

In [ ]:
save_media(combined_afmedium_d, "combined_af7_c1i127", media_dir)

## 3. Analysis of the completed medium

In [22]:
# compare complete media from the communities
only1 = set(af_c1.keys()) - set(af_c2.keys())
only2 = set(af_c2.keys()) - set(af_c1.keys())

print(only1)
print(only2)

{'EX_R_3httdca_m', 'EX_nmn_m', 'EX_tyr__L_m', 'EX_uacgam_m', 'EX_acmana_m'}
{'EX_glygln_m', 'EX_asn__L_m', 'EX_cu_m'}


In [23]:
# compare base medium to completed media 
d1 = set(AF7_s.index) - set(af_c1.keys())
d2 = set(AF7_s.index) - set(af_c2.keys())

In [ ]:
# composition of completed medium
as_d, c_d = assign_metaboliteclass(combined_afmedium, met_class_dict)

In [22]:
c_d

defaultdict(int,
            {'Organic Acids & Carboxylates': 7,
             'Carbohydrates, Sugars & Derivatives': 9,
             'Amino Acids, Peptides & Polyamines': 16,
             'Inorganic Compounds, Ions & Gases': 19,
             'Fatty Acids, Lipids & Steroids': 1})